# ETL-15 · Datas de sinistro e óbito

- **Etapa:** Conversão e validação de datas
- **Responsável:** Samuel (@safn007)
- **Issue:** #17
- **Entrada:** `intermediarios/01_bruto.parquet`
- **Saída:** `intermediarios/04_datas.parquet`

## 1. Setup

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display

RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / ".git").exists())

DADOS = RAIZ / "data"
INTERM = DADOS / "intermediarios"
INTERM.mkdir(parents=True, exist_ok=True)

ENTRADA = INTERM / "01_bruto.parquet"
SAIDA = INTERM / "04_datas.parquet"

## 2. Carregar a entrada

In [2]:
df = pd.read_parquet(ENTRADA)

print("shape:", df.shape)
print("colunas:", df.columns.tolist())
df.head()

shape: (806977, 33)
colunas: ['id_sinistro', 'id_pessoa', 'id_veiculo', 'cod_ibge', 'municipio', 'regiao_administrativa', 'tipo_via', 'tipo_veiculo_vitima', 'tipo_de_vitima', 'modo_transporte_vitima', 'sexo', 'idade', 'gravidade_lesao', 'faixa_etaria_demografica', 'faixa_etaria_legal', 'profissao', 'grau_de_instrucao', 'nacionalidade', 'data_sinistro', 'ano_sinistro', 'mes_sinistro', 'dia_sinistro', 'ano_mes_sinistro', 'data_obito', 'ano_obito', 'mes_obito', 'dia_obito', 'ano_mes_obito', 'local_obito', 'local_via', 'tempo_sinistro_obito', 'ano_origem', 'arquivo_origem']


,id_sinistro,id_pessoa,id_veiculo,cod_ibge,municipio,regiao_administrativa,tipo_via,tipo_veiculo_vitima,tipo_de_vitima,modo_transporte_vitima,...,data_obito,ano_obito,mes_obito,dia_obito,ano_mes_obito,local_obito,local_via,tempo_sinistro_obito,ano_origem,arquivo_origem
0,2240338,1979797,2874666.0,3552502,SUZANO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,AUTOMOVEL,PASSAGEIRO,AUTOMOVEL,...,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,2021,pessoas_01-2021.csv
1,2240338,1979798,2874666.0,3552502,SUZANO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,AUTOMOVEL,CONDUTOR,AUTOMOVEL,...,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,2021,pessoas_01-2021.csv
2,1447800,1284984,NaN,3550308,SAO PAULO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,NaN,PEDESTRE,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,2021,pessoas_01-2021.csv
3,1783410,563279,2648403.0,3550308,SAO PAULO,METROPOLITANA DE SÃO PAULO,VIAS URBANAS,AUTOMOVEL,CONDUTOR,AUTOMOVEL,...,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,2021,pessoas_01-2021.csv
4,1798856,687322,NaN,3547601,SANTA ROSA DE VITERBO,RIBEIRÃO PRETO,VIAS URBANAS,NaN,PASSAGEIRO,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,PUBLICO,NaN,2021,pessoas_01-2021.csv


## 3. Diagnóstico ANTES

In [3]:
def diagnostico(df: pd.DataFrame) -> pd.DataFrame:
    """Resume o tipo, o percentual de nulos e os valores distintos por coluna."""
    return pd.DataFrame({
        "dtype": df.dtypes.astype(str),
        "pct_nulo": (df.isna().mean() * 100).round(2),
        "n_distintos": df.nunique(),
    })

antes = diagnostico(df)
antes

,dtype,pct_nulo,n_distintos
id_sinistro,int64,0.00,402695
id_pessoa,int64,0.00,806977
id_veiculo,float64,21.04,554294
cod_ibge,int64,0.00,645
municipio,str,0.00,645
regiao_administrativa,str,0.00,16
tipo_via,str,0.00,3
tipo_veiculo_vitima,str,20.98,7
tipo_de_vitima,str,0.00,4
modo_transporte_vitima,str,20.54,8


## 4. Transformações

### 4A. - Configuração e conversão

In [4]:
# As chaves são as datas completas. As tuplas seguem a ordem
# (coluna de ano, coluna de mês, coluna de dia, coluna de ano/mês).
CAMPOS = {
    "data_sinistro": ("ano_sinistro", "mes_sinistro", "dia_sinistro", "ano_mes_sinistro"),
    "data_obito": ("ano_obito", "mes_obito", "dia_obito", "ano_mes_obito"),
}
DATA_SINISTRO, DATA_OBITO = CAMPOS
DERIVADAS = [coluna for grupo in CAMPOS.values() for coluna in grupo]

if len(set(DERIVADAS)) != 8 or set(CAMPOS) & set(DERIVADAS):
    raise ValueError("Confira se há oito derivadas distintas das duas datas completas.")

faltantes = set(CAMPOS).union(DERIVADAS) - set(df.columns)
if faltantes:
    raise KeyError(f"Confira os nomes no resultado da seção 2. Faltam: {sorted(faltantes)}")


def limpar_texto(serie: pd.Series) -> pd.Series:
    """Remove espaços e considera campos vazios como ausentes."""
    return serie.astype("string").str.strip().replace("", pd.NA)


def converter_datas(df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """Converte datas e distingue valores ausentes de datas inválidas."""
    resultado = df.copy()
    auditoria = []

    for coluna in CAMPOS:
        original = df[coluna]
        texto = limpar_texto(original)

        if pd.api.types.is_datetime64_any_dtype(original.dtype):
            data = original.dt.normalize()
        else:
            data = pd.to_datetime(texto, format="%d/%m/%Y", errors="coerce")
            iso = pd.to_datetime(texto, format="%Y-%m-%d", errors="coerce")
            data = data.fillna(iso)

        auditoria.append({
            "coluna": coluna,
            "ausentes": int(texto.isna().sum()),
            "preenchidas_inválidas": int((texto.notna() & data.isna()).sum()),
        })
        resultado[coluna] = data

    return resultado, pd.DataFrame(auditoria).set_index("coluna")

### 4B. - Conferência e cálculo

In [5]:
def normalizar_ano_mes(texto: pd.Series) -> pd.Series:
    """Transforma AAAA/MM ou MM/AAAA em AAAAMM para comparação."""
    ano_primeiro = texto.str.extract(r"^(\d{4})[/-](\d{1,2})$")
    mes_primeiro = texto.str.extract(r"^(\d{1,2})[/-](\d{4})$")

    ano = pd.to_numeric(ano_primeiro[0].fillna(mes_primeiro[1]), errors="coerce")
    mes = pd.to_numeric(ano_primeiro[1].fillna(mes_primeiro[0]), errors="coerce")
    return (ano * 100 + mes).where(mes.between(1, 12))


def conferir_derivadas(df: pd.DataFrame) -> pd.DataFrame:
    """Compara as derivadas originais com as datas já convertidas."""
    auditoria = []

    for coluna_data, derivadas in CAMPOS.items():
        data = df[coluna_data]
        esperados = (
            data.dt.year,
            data.dt.month,
            data.dt.day,
            data.dt.year * 100 + data.dt.month,
        )
        limites = ((1, 9999), (1, 12), (1, 31), None)

        for coluna, esperado, limite in zip(derivadas, esperados, limites):
            texto = limpar_texto(df[coluna])

            if limite is None:
                informado = normalizar_ano_mes(texto)
            else:
                informado = pd.to_numeric(texto, errors="coerce")
                valido = informado.between(*limite) & informado.mod(1).eq(0)
                informado = informado.where(valido)

            comparavel = data.notna() & informado.notna()
            divergente = comparavel & informado.ne(esperado)

            auditoria.append({
                "coluna": coluna,
                "comparáveis": int(comparavel.sum()),
                "divergências": int(divergente.sum()),
                "derivada_ausente_com_data_válida": int((texto.isna() & data.notna()).sum()),
                "derivada_preenchida_inválida": int((texto.notna() & informado.isna()).sum()),
                "derivada_preenchida_sem_data_válida": int((texto.notna() & data.isna()).sum()),
            })

    return pd.DataFrame(auditoria).set_index("coluna")


def calcular_dias_ate_obito(df: pd.DataFrame) -> pd.DataFrame:
    """Calcula a diferença em dias corridos e preserva os intervalos não calculáveis."""
    resultado = df.copy()
    intervalo = resultado[DATA_OBITO] - resultado[DATA_SINISTRO]
    resultado["dias_ate_obito"] = intervalo.dt.days.astype("Int64")
    return resultado


def remover_derivadas(df: pd.DataFrame) -> pd.DataFrame:
    """Remove somente as oito colunas derivadas já conferidas."""
    return df.drop(columns=DERIVADAS)

### 4C. - Execução e evidências

In [6]:
n_antes = len(df)
colunas_antes = set(df.columns)

df, auditoria_datas = converter_datas(df)
auditoria_derivadas = conferir_derivadas(df)
df = calcular_dias_ate_obito(df)

display(auditoria_datas)
display(auditoria_derivadas)

dias = df["dias_ate_obito"]
obito_sem_sinistro = df[DATA_OBITO].notna() & df[DATA_SINISTRO].isna()
sinistro_fora_do_periodo = (
    df[DATA_SINISTRO].notna() & ~df[DATA_SINISTRO].dt.year.isin([2021, 2022, 2023])
)

regras = pd.Series({
    "sinistros fora de 2021–2023": int(sinistro_fora_do_periodo.sum()),
    "óbitos com data, mas sem data de sinistro válida": int(obito_sem_sinistro.sum()),
    "óbitos anteriores ao sinistro": int(dias.lt(0).sum()),
    "óbitos no mesmo dia": int(dias.eq(0).sum()),
    "intervalos não calculáveis": int(dias.isna().sum()),
}, name="linhas")

display(regras.to_frame())

df = remover_derivadas(df)

assert len(df) == n_antes
assert colunas_antes - set(df.columns) == set(DERIVADAS)
assert {DATA_SINISTRO, DATA_OBITO, "dias_ate_obito"} <= set(df.columns)

print(f"linhas: {n_antes} -> {len(df)} (removidas: {n_antes - len(df)})")
print("colunas removidas:", DERIVADAS)

,ausentes,preenchidas_inválidas
coluna,,
data_sinistro,0,0
data_obito,791184,0


,comparáveis,divergências,derivada_ausente_com_data_válida,derivada_preenchida_inválida,derivada_preenchida_sem_data_válida
coluna,,,,,
ano_sinistro,806977,0,0,0,0
mes_sinistro,806977,0,0,0,0
dia_sinistro,806977,0,0,0,0
ano_mes_sinistro,806977,0,0,0,0
ano_obito,15793,0,0,0,0
mes_obito,15793,0,0,0,0
dia_obito,15793,0,0,0,0
ano_mes_obito,15793,0,0,0,0


,linhas
sinistros fora de 2021–2023,0
"óbitos com data, mas sem data de sinistro válida",0
óbitos anteriores ao sinistro,0
óbitos no mesmo dia,12805
intervalos não calculáveis,791184


linhas: 806977 -> 806977 (removidas: 0)
colunas removidas: ['ano_sinistro', 'mes_sinistro', 'dia_sinistro', 'ano_mes_sinistro', 'ano_obito', 'mes_obito', 'dia_obito', 'ano_mes_obito']


## 5. Diagnóstico DEPOIS (mesma tabela do passo 3)

In [7]:
depois = diagnostico(df)
depois

,dtype,pct_nulo,n_distintos
id_sinistro,int64,0.00,402695
id_pessoa,int64,0.00,806977
id_veiculo,float64,21.04,554294
cod_ibge,int64,0.00,645
municipio,str,0.00,645
regiao_administrativa,str,0.00,16
tipo_via,str,0.00,3
tipo_veiculo_vitima,str,20.98,7
tipo_de_vitima,str,0.00,4
modo_transporte_vitima,str,20.54,8


## 6. Salvar a saída

In [8]:
df.to_parquet(SAIDA, index=False)
print("salvo em:", SAIDA, "| shape:", df.shape)

salvo em: c:\Users\samue\Documents\Projects\Projeto_BD\Integracao_BD\data\intermediarios\04_datas.parquet | shape: (806977, 26)


## 7. Resumo

A ETL-15 converteu as datas de sinistro e óbito e conferiu as oito colunas derivadas de ano, mês, dia e ano/mês. As **806.977 datas de sinistro** eram válidas e estavam dentro do período de 2021 a 2023. Entre os **15.793 registros com data de óbito preenchida**, não houve datas inválidas nem óbitos anteriores ao sinistro.

Não foram encontradas divergências entre as datas completas e suas derivadas nas linhas comparáveis. Após a conferência, as oito colunas derivadas foram removidas por serem redundantes. Nenhuma linha foi excluída.

A coluna `dias_ate_obito` foi calculada para **15.793 registros**. Em **12.805**, o óbito ocorreu no mesmo dia do sinistro e o intervalo foi zero. Nos outros **2.988**, o óbito ocorreu em data posterior. Para os **791.184 registros sem data de óbito**, o intervalo permaneceu nulo.